# YOLO11s + Inner-IoU Training on Kaggle

这版 notebook 已经改成 `baseline + Inner-IoU` 的训练版本。

- 默认使用 `yolo11s.pt` 作为预训练权重
- 训练脚本是 `train_yolo11s_inner_iou.py`
- 只修改边界框回归损失，模型结构保持 `yolo11s baseline`


In [ ]:
# Cell 1: 克隆代码
%cd /kaggle/working
!rm -rf yolov11s_kuangjing
!git clone https://github.com/tuanziya666/yolov11s_kuangjing.git
%cd /kaggle/working/yolov11s_kuangjing
!git pull
!pip install -q -e .


In [ ]:
# Cell 2: 环境检查并关闭 wandb/raytune
%cd /kaggle/working/yolov11s_kuangjing
!python -c "import torch, cv2, numpy; from ultralytics import YOLO; print('OK'); print(torch.__version__, cv2.__version__, numpy.__version__)"
!python -c "from ultralytics.utils import SETTINGS; SETTINGS.update(wandb=False, raytune=False); print('wandb=', SETTINGS['wandb']); print('raytune=', SETTINGS['raytune'])"


In [ ]:
# Cell 3: 查看 Kaggle 数据集路径
!find /kaggle/input -maxdepth 3 -type d | head -n 50


In [ ]:
# Cell 4: 写 Kaggle 专用数据集配置
%%writefile /kaggle/working/yolov11s_kuangjing/ultralytics/cfg/datasets/coalmine4_kaggle.yaml
path: /kaggle/input/datasets/yuanfangshang/kuangjing
train: images/train
val: images/val
test: images/test

names:
  0: chuck
  1: gripper
  2: drill_pipe
  3: coal_miner


In [ ]:
# Cell 5: 确认数据配置和 Inner-IoU 代码文件
%cd /kaggle/working/yolov11s_kuangjing
!cat ultralytics/cfg/datasets/coalmine4_kaggle.yaml
!ls -lh train_yolo11s_inner_iou.py ultralytics/utils/metrics.py ultralytics/utils/loss.py


In [ ]:
# Cell 6: 训练 baseline + Inner-IoU
%cd /kaggle/working/yolov11s_kuangjing
!python -u train_yolo11s_inner_iou.py \
  --data ultralytics/cfg/datasets/coalmine4_kaggle.yaml \
  --name yolo11s_inner_iou_kaggle \
  --project /kaggle/working/runs \
  --device 0 \
  --batch 16 \
  --workers 2 \
  --epochs 100 \
  --inner-ratio 0.8 \
  --imgsz 640


In [ ]:
# Cell 7: 查看训练结果
!ls /kaggle/working/runs/yolo11s_inner_iou_kaggle
!ls /kaggle/working/runs/yolo11s_inner_iou_kaggle/weights


In [ ]:
# Cell 8: 打包结果，方便从 Kaggle 下载
!zip -r /kaggle/working/yolo11s_inner_iou_kaggle.zip /kaggle/working/runs/yolo11s_inner_iou_kaggle
